<style>
    :root {
        --accent: #38bdf8;
        --bg-dark: #0f172a;
    }
    .jp-Notebook { background-color: var(--bg-dark) !important; color: #e2e8f0 !important; }
    .jp-RenderedHTMLCommon h1, .jp-RenderedHTMLCommon h2 { color: var(--accent) !important; border-bottom: 1px solid rgba(56, 189, 248, 0.2) !important; }
    .insight-card {
        background: rgba(56, 189, 248, 0.05);
        border-left: 4px solid var(--accent);
        padding: 1.2rem;
        border-radius: 8px;
        margin: 1rem 0;
    }
    .status-tag {
        background: var(--accent);
        color: var(--bg-dark);
        padding: 2px 8px;
        border-radius: 4px;
        font-weight: bold;
        font-size: 0.8em;
    }
</style>

# 🔬 Intel Core Ultra NPU: Unified GNN Benchmarking Suite

<span class="status-tag">ADVANCED ANALYTICS MODE</span>

This notebook provides a complete research pipeline with time-synchronized energy correlation for each model architecture.

In [ ]:
import os, sys, json, shutil
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display, Markdown, HTML

RESULTS_DIR = Path("results")
MODELS_DIR = Path("models")
FIGURES_DIR = Path("paper/figures")

plt.style.use('dark_background')
sns.set_theme(style="darkgrid", palette="muted")
print("✅ Advanced Analytics Ready.")

### 📦 Step 1: Model Suite Generation
GNN + baseline (CNN/NLP) models; standardized and quantized.

In [ ]:
!python analysis/model_prep.py

### 📈 Step 2: Scalability & Timing Study
Executing models while recording high-precision start/end timestamps for energy correlation.

In [ ]:
!python analysis/scalability_analyzer.py --iterations 100 --repeats 3

### 🔍 Step 3: Operator Profiling
Deep-dive into graph fusion and bottleneck identification.

In [ ]:
!python analysis/profiling_analyzer.py

### 💻 Step 4: Hardware Comparison
Benchmarking all models across CPU, GPU, and NPU.

In [ ]:
import glob
from pathlib import Path

# Choose mode: "quick" or "full"
RUN_MODE = "quick"

if RUN_MODE == "quick":
    iterations = 10
    repeats = 1
    allowlist = {
        "gcn_fp32.onnx",
        "graphsage_fp32.onnx",
        "resnet50_fp32.onnx",
        "mobilenetv2_fp32.onnx",
        "bert-tiny_fp32.onnx",
    }
else:
    iterations = 100
    repeats = 3
    allowlist = None

model_paths = [Path(p) for p in sorted(glob.glob("models/*_fp32.onnx"))]
found = {p.name.lower() for p in model_paths}
if allowlist:
    missing = sorted(name for name in allowlist if name not in found)
    if missing:
        print(f"⚠️ Missing models: {', '.join(missing)}")
for model_path in model_paths:
    if allowlist and model_path.name.lower() not in allowlist:
        continue
    print(f"🚀 Device Comparison: {model_path}")
    !python analysis/hw_comparison.py --model {model_path} --iterations {iterations} --repeats {repeats}

### 🔋 Step 5: Energy Correlation Analysis
Correlating HWiNFO sensor logs with model timestamps to extract precise power metrics per architecture.

In [ ]:
LOG_FILE = "results/hwinfo_log.csv"
MATRIX_FILE = "results/scalability_matrix.csv"

if Path(MATRIX_FILE).exists():
    print("🔗 Running Energy Analysis (using fallback estimation if logs missing)...")
    # We pass the log file path even if it doesn't exist, the script handles the fallback
    !python analysis/energy_analyzer.py --log {LOG_FILE} --matrix {MATRIX_FILE}
else:
    print("ℹ️ No scalability matrix found. Run Step 2 first.")

### 📂 Step 6: Artifact Sync
Exporting all plots to paper directory.

In [ ]:
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
for png in RESULTS_DIR.rglob("*.png"):
    shutil.copy2(png, FIGURES_DIR / png.name)
print(f"✅ Sync complete: {FIGURES_DIR}")

## 📊 Research Dashboard

In [ ]:
if Path(MATRIX_FILE).exists():
    df = pd.read_csv(MATRIX_FILE)
    
    fig, ax1 = plt.subplots(figsize=(12, 6))
    sns.barplot(data=df, x="model", y="speedup", hue="model", palette="viridis", legend=False, ax=ax1, alpha=0.7)
    ax1.set_ylabel("Speedup Factor (x)", fontweight='bold')
    ax1.axhline(1.0, color="red", ls="--")
    
    if "avg_npu_power_w" in df.columns:
        ax2 = ax1.twinx()
        sns.lineplot(data=df, x="model", y="avg_npu_power_w", color="#facc15", marker="o", ax=ax2, label="Avg Power (W)")
        ax2.set_ylabel("NPU Power Consumption (W)", color="#facc15", fontweight='bold')
    
    plt.title("NPU Performance vs Energy consumption", fontsize=14)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    # Artifacts
    for img in ["roofline_model.png", "latency_stacked_100pct.png"]:
        if (RESULTS_DIR / img).exists():
            display(Markdown(f"### 📍 {img.upper()}"))
            display(Image(filename=str(RESULTS_DIR / img), width=850))
else:
    print("❌ No data found.")
